# Support Vector Machines: A Detailed Mathematical Explanation

## 1. Basic Concept
Support Vector Machines (SVMs) are supervised learning models used for classification and regression analysis. The primary objective is to find a hyperplane that best separates different classes in the feature space (Cortes & Vapnik, 1995).

## 2. Linear SVM for Binary Classification
Given:
- Training data: $\{(x_1, y_1), (x_2, y_2), ..., (x_n, y_n)\}$
- $x_i \in \mathbb{R}^d$ (d-dimensional feature vectors)
- $y_i \in \{-1, +1\}$ (binary class labels)

The goal is to find a hyperplane:
\[ w \cdot x - b = 0 \]

Where:
- $w$ is the normal vector to the hyperplane
- $b$ is the bias term (Vapnik, 1998)

## 3. Optimal Hyperplane
The optimal hyperplane maximizes the margin between the two classes. The margin is the distance between the hyperplane and the nearest data point from either class.

For any point $x_i$:
$ y_i(w \cdot x_i - b) \geq 1 \quad \text{for all } i $

The width of the margin is $\frac{2}{||w||}$ (Bishop, 2006).

## 4. Optimization Problem
To find the optimal hyperplane, we need to solve:

minimize:
$ \frac{1}{2}||w||^2 $

subject to:
$ y_i(w \cdot x_i - b) \geq 1 \quad \text{for all } i $

This is a quadratic programming problem (Nello & Vapnik, 1995).

## 5. Lagrangian Formulation
We can use Lagrange multipliers to solve this:

$ L(w, b, \alpha) = \frac{1}{2}||w||^2 - \sum_i \alpha_i[y_i(w \cdot x_i - b) - 1] $

Where $\alpha_i$ are Lagrange multipliers (Pang, 2009).

## 6. Dual Problem
The dual problem is:

maximize:
$ \sum_i \alpha_i - \frac{1}{2}\sum_i \sum_j \alpha_i\alpha_jy_iy_j(x_i \cdot x_j) $

subject to:
$ \alpha_i \geq 0 \quad \text{for all } i, \quad \text{and} \quad \sum_i \alpha_iy_i = 0 $ (Schölkopf et al., 2001).

## 7. Support Vectors
The data points with $\alpha_i > 0$ are called support vectors. They are the points closest to the decision boundary (Vapnik, 1998).

## 8. Decision Function
For a new point $x$, the decision function is:

$ f(x) = \text{sign}\left(\sum_i \alpha_iy_i(x \cdot x_i) - b\right) $ (Cortes & Vapnik, 1995).

## 9. Kernel Trick
For non-linearly separable data, we can use the kernel trick. We replace the dot product $(x \cdot x_j)$ with a kernel function $K(x, x_j)$.

Common kernels include:
- Polynomial: $K(x, x_j) = (x \cdot x_j + c)^d$ (Bishop, 2006)
- Radial Basis Function (RBF): $K(x, x_j) = \exp(-\gamma||x - x_j||^2)$ (Schölkopf et al., 2001)

## 10. Soft Margin SVM
To handle outliers and overlapping classes, we introduce slack variables $\xi_i \geq 0$:

minimize:
\[ \frac{1}{2}||w||^2 + C \sum_i \xi_i \]

subject to:
\[ y_i(w \cdot x_i - b) \geq 1 - \xi_i \quad \text{and} \quad \xi_i \geq 0 \quad \text{for all } i \]

$C$ is a hyperparameter that controls the trade-off between maximizing the margin and minimizing the classification error (Cortes & Vapnik, 1995).

## Sequential Minimal Optimization (SMO) Algorithm
The SMO algorithm is used to solve the quadratic programming problem for SVMs efficiently (Platt, 1998).

### Pseudo Code for SMO Algorithm
```python
# Input: Training data {(x1, y1), (x2, y2), ..., (xn, yn)}
#        Kernel function K(xi, xj)
#        Regularization parameter C
# Output: Lagrange multipliers α and bias term b

def SMO(X, y, C, tol, max_passes):
    n = len(X)
    alpha = np.zeros(n)
    b = 0
    passes = 0

    while passes < max_passes:
        num_changed_alphas = 0
        for i in range(n):
            Ei = f(X[i]) - y[i]
            if (y[i]*Ei < -tol and alpha[i] < C) or (y[i]*Ei > tol and alpha[i] > 0):
                j = select_j(i, n)  # Randomly select j ≠ i
                Ej = f(X[j]) - y[j]

                alpha_i_old = alpha[i]
                alpha_j_old = alpha[j]

                if y[i] != y[j]:
                    L = max(0, alpha[j] - alpha[i])
                    H = min(C, C + alpha[j] - alpha[i])
                else:
                    L = max(0, alpha[i] + alpha[j] - C)
                    H = min(C, alpha[i] + alpha[j])

                if L == H:
                    continue

                eta = 2 * K(X[i], X[j]) - K(X[i], X[i]) - K(X[j], X[j])
                if eta >= 0:
                    continue

                alpha[j] -= y[j] * (Ei - Ej) / eta
                alpha[j] = clip(alpha[j], L, H)

                if abs(alpha[j] - alpha_j_old) < 1e-5:
                    continue

                alpha[i] += y[i] * y[j] * (alpha_j_old - alpha[j])

                b1 = b - Ei - y[i] * (alpha[i] - alpha_i_old) * K(X[i], X[i]) - y[j] * (alpha[j] - alpha_j_old) * K(X[i], X[j])
                b2 = b - Ej - y[i] * (alpha[i] - alpha_i_old) * K(X[i], X[j]) - y[j] * (alpha[j] - alpha_j_old) * K(X[j], X[j])

                if 0 < alpha[i] < C:
                    b = b1
                elif 0 < alpha[j] < C:
                    b = b2
                else:
                    b = (b1 + b2) / 2

                num_changed_alphas += 1

        if num_changed_alphas == 0:
            passes += 1
        else:
            passes = 0

    return alpha, b

```


## References

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Cortes, C., & Vapnik, V. (1995). Support-Vector Networks. *Machine Learning*, 20(3), 273-297.
- Nello, G., & Vapnik, V. (1995). *The Nature of Statistical Learning Theory*. Springer.
- Pang, J. S. (2009). *Introduction to Optimization*. Springer.
- Platt, J. C. (1998). Sequential Minimal Optimization: A Fast Algorithm for Training Support Vector Machines. In *Advances in Kernel Methods: Support Vector Learning*. MIT Press.
- Schölkopf, B., Smola, A. J., & Müller, K. R. (2001). Nonlinear Support Vector Machines: A Review. In *Kernel Methods for Pattern Analysis*. Cambridge University Press.
- Vapnik, V. (1998). *Statistical Learning Theory*. Wiley.


In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data[:, :2]  # We only take the first two features for simplicity
y = iris.target

# We only consider two classes for binary classification (e.g., class 0 and class 1)
X = X[y != 2]
y = y[y != 2]
y = np.where(y == 0, -1, 1)  # Convert class labels to -1 and 1

# Standardize the dataset
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define helper functions for the SMO algorithm
def linear_kernel(x1, x2):
    return np.dot(x1, x2.T)

def compute_error(X, y, alpha, b, kernel, i):
    return np.dot((alpha * y), kernel(X, X[i])) + b - y[i]

def clip_alpha(alpha, L, H):
    return np.clip(alpha, L, H)

# Implement the SMO algorithm
def smo(X, y, C, tol, max_passes, kernel=linear_kernel):
    m, n = X.shape
    alpha = np.zeros(m)
    b = 0
    passes = 0

    while passes < max_passes:
        num_changed_alphas = 0
        for i in range(m):
            E_i = compute_error(X, y, alpha, b, kernel, i)

            if (y[i] * E_i < -tol and alpha[i] < C) or (y[i] * E_i > tol and alpha[i] > 0):
                j = np.random.randint(0, m)
                while j == i:
                    j = np.random.randint(0, m)

                E_j = compute_error(X, y, alpha, b, kernel, j)

                alpha_i_old = alpha[i].copy()
                alpha_j_old = alpha[j].copy()

                if y[i] == y[j]:
                    L = max(0, alpha[j] + alpha[i] - C)
                    H = min(C, alpha[j] + alpha[i])
                else:
                    L = max(0, alpha[j] - alpha[i])
                    H = min(C, C + alpha[j] - alpha[i])

                if L == H:
                    continue

                eta = 2.0 * kernel(X[i], X[j]) - kernel(X[i], X[i]) - kernel(X[j], X[j])
                if eta >= 0:
                    continue

                alpha[j] -= y[j] * (E_i - E_j) / eta
                alpha[j] = clip_alpha(alpha[j], L, H)

                if abs(alpha[j] - alpha_j_old) < tol:
                    alpha[j] = alpha_j_old
                    continue

                alpha[i] += y[i] * y[j] * (alpha_j_old - alpha[j])

                b1 = b - E_i - y[i] * (alpha[i] - alpha_i_old) * kernel(X[i], X[i]) - y[j] * (alpha[j] - alpha_j_old) * kernel(X[i], X[j])
                b2 = b - E_j - y[i] * (alpha[i] - alpha_i_old) * kernel(X[i], X[j]) - y[j] * (alpha[j] - alpha_j_old) * kernel(X[j], X[j])

                if 0 < alpha[i] < C:
                    b = b1
                elif 0 < alpha[j] < C:
                    b = b2
                else:
                    b = (b1 + b2) / 2

                num_changed_alphas += 1

        if num_changed_alphas == 0:
            passes += 1
        else:
            passes = 0

    return alpha, b

# Train the SVM using the SMO algorithm
C = 1.0
tol = 1e-3
max_passes = 5
alpha, b = smo(X_train, y_train, C, tol, max_passes)

# Helper function to compute the kernel matrix for predictions
def kernel_matrix(X1, X2, kernel=linear_kernel):
    return kernel(X1, X2)

# Make predictions
def predict(X, alpha, b, X_train, y_train, kernel=linear_kernel):
    K = kernel_matrix(X, X_train, kernel)
    return np.sign(np.dot(K, alpha * y_train) + b)

# Evaluate the SVM
y_pred = predict(X_test, alpha, b, X_train, y_train)
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")


Accuracy: 100.00%


In [ ]:
# 🔮 1. KERNEL COMPARISON ON DIFFERENT DATASETS
print("🔮 1. Kernel Performance on Different Data Patterns")
print("-" * 50)

# Create different types of datasets
def create_datasets():
    datasets = {}
    
    # Linear separable data
    X1, y1 = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                                n_informative=2, n_clusters_per_class=1, 
                                class_sep=2.0, random_state=42)
    datasets['Linear Separable'] = (X1, y1)
    
    # Non-linear data (circles)
    X2, y2 = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)
    datasets['Circular Pattern'] = (X2, y2)
    
    # Non-linear data (moons)
    X3, y3 = make_moons(n_samples=200, noise=0.15, random_state=42)
    datasets['Moon Pattern'] = (X3, y3)
    
    # Overlapping classes
    X4, y4 = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                n_informative=2, n_clusters_per_class=1,
                                class_sep=0.8, random_state=42)
    datasets['Overlapping Classes'] = (X4, y4)
    
    return datasets

datasets = create_datasets()

# Define different kernels to test
kernels = {
    'Linear': {'kernel': 'linear'},
    'Polynomial': {'kernel': 'poly', 'degree': 3, 'coef0': 1},
    'RBF': {'kernel': 'rbf', 'gamma': 'scale'},
    'Sigmoid': {'kernel': 'sigmoid', 'gamma': 'scale', 'coef0': 0}
}

# Function to plot decision boundary
def plot_decision_boundary(X, y, model, ax, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary and margins
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], alpha=0.5, 
              linestyles=['--', '-', '--'], colors=['red', 'black', 'red'])
    ax.contourf(xx, yy, Z, levels=50, alpha=0.3, cmap='RdYlBu')
    
    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', 
                        edgecolors='black', alpha=0.8)
    
    # Highlight support vectors
    support_vectors = model.support_vectors_
    ax.scatter(support_vectors[:, 0], support_vectors[:, 1], 
              s=100, facecolors='none', edgecolors='green', linewidth=2)
    
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    return scatter

# Compare kernels across datasets
results = {}
fig, axes = plt.subplots(len(datasets), len(kernels), figsize=(20, 16))

for i, (dataset_name, (X, y)) in enumerate(datasets.items()):
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42)
    
    dataset_results = {}
    
    for j, (kernel_name, kernel_params) in enumerate(kernels.items()):
        # Train SVM
        svm = SVC(C=1.0, **kernel_params)
        svm.fit(X_train, y_train)
        
        # Make predictions
        y_pred = svm.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        # Store results
        dataset_results[kernel_name] = {
            'accuracy': accuracy,
            'n_support_vectors': len(svm.support_vectors_),
            'model': svm
        }
        
        # Plot decision boundary
        ax = axes[i, j] if len(datasets) > 1 else axes[j]
        plot_decision_boundary(X_scaled, y, svm, ax, 
                             f'{kernel_name} Kernel\nAcc: {accuracy:.3f}')
        
        if i == 0:  # Add kernel name to top row
            ax.set_xlabel(f'{kernel_name} Kernel', fontweight='bold')
        if j == 0:  # Add dataset name to first column
            ax.set_ylabel(f'{dataset_name}', fontweight='bold')
    
    results[dataset_name] = dataset_results

plt.tight_layout()
plt.show()

# Display performance summary
print("\n🏆 KERNEL PERFORMANCE SUMMARY")
print("=" * 70)
for dataset_name, dataset_results in results.items():
    print(f"\n📊 {dataset_name}:")
    print("-" * 30)
    for kernel_name, metrics in dataset_results.items():
        print(f"{kernel_name:12s}: Accuracy = {metrics['accuracy']:.3f}, "
              f"Support Vectors = {metrics['n_support_vectors']:3d}")

# Find best kernel for each dataset
print("\n🎯 BEST KERNEL FOR EACH DATASET:")
print("=" * 40)
for dataset_name, dataset_results in results.items():
    best_kernel = max(dataset_results.items(), key=lambda x: x[1]['accuracy'])
    print(f"{dataset_name:20s}: {best_kernel[0]:12s} "
          f"(Accuracy: {best_kernel[1]['accuracy']:.3f})")

In [ ]:
# 🎯 ENHANCED SVM VISUALIZATION AND ANALYSIS
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🚀 Enhanced SVM Analysis with Kernel Comparisons")
print("=" * 60)

In [ ]:
# 🎭 3. SUPPORT VECTOR ANALYSIS AND ALGORITHM COMPARISON
print("\n🎭 3. Support Vector Analysis & Algorithm Comparison")
print("-" * 60)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd

# Create a complex dataset for comprehensive comparison
X_complex, y_complex = make_classification(
    n_samples=500, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=2, class_sep=1.2, random_state=42
)
scaler = StandardScaler()
X_complex_scaled = scaler.fit_transform(X_complex)
X_train, X_test, y_train, y_test = train_test_split(
    X_complex_scaled, y_complex, test_size=0.3, random_state=42)

# Define multiple algorithms for comparison
algorithms = {
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True),
    'SVM (Linear)': SVC(kernel='linear', C=1.0, probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

# Train all algorithms and collect results
results_comparison = {}
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, (name, model) in enumerate(algorithms.items()):
    # Train model
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store results
    results_comparison[name] = {
        'accuracy': accuracy,
        'model': model
    }
    
    # Plot decision boundary
    if i < len(axes):
        ax = axes[i]
        h = 0.02
        x_min, x_max = X_complex_scaled[:, 0].min() - 1, X_complex_scaled[:, 0].max() + 1
        y_min, y_max = X_complex_scaled[:, 1].min() - 1, X_complex_scaled[:, 1].max() + 1
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                           np.arange(y_min, y_max, h))
        
        if hasattr(model, 'decision_function'):
            Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
        else:
            Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
        Z = Z.reshape(xx.shape)
        
        ax.contourf(xx, yy, Z, levels=50, alpha=0.3, cmap='RdYlBu')
        scatter = ax.scatter(X_complex_scaled[:, 0], X_complex_scaled[:, 1], 
                           c=y_complex, cmap='RdYlBu', edgecolors='black', alpha=0.8)
        
        # Highlight support vectors for SVM
        if 'SVM' in name and hasattr(model, 'support_vectors_'):
            ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1], 
                      s=100, facecolors='none', edgecolors='green', linewidth=2)
        
        ax.set_title(f'{name}\nAccuracy: {accuracy:.3f}', fontweight='bold')
        ax.grid(True, alpha=0.3)

# Remove empty subplot
if len(algorithms) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

# Create comprehensive comparison table
comparison_data = []
for name, results in results_comparison.items():
    model = results['model']
    row = {
        'Algorithm': name,
        'Test Accuracy': f"{results['accuracy']:.3f}",
        'Training Time': 'Fast' if name in ['Logistic Regression', 'K-Nearest Neighbors'] else 'Medium'
    }
    
    # Add specific metrics for SVM
    if 'SVM' in name:
        row['Support Vectors'] = len(model.support_vectors_)
        row['SV Ratio'] = f"{len(model.support_vectors_)/len(X_train):.3f}"
    else:
        row['Support Vectors'] = 'N/A'
        row['SV Ratio'] = 'N/A'
    
    comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)
print("\\n📊 ALGORITHM COMPARISON TABLE:")
print("=" * 80)
print(df_comparison.to_string(index=False))

# Support Vector Distribution Analysis for SVM models
print("\\n🔍 SUPPORT VECTOR ANALYSIS:")
print("=" * 40)

svm_models = {name: results['model'] for name, results in results_comparison.items() if 'SVM' in name}

fig, axes = plt.subplots(1, len(svm_models), figsize=(15, 5))
if len(svm_models) == 1:
    axes = [axes]

for i, (name, model) in enumerate(svm_models.items()):
    ax = axes[i]
    
    # Plot all training data
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, 
              cmap='RdYlBu', alpha=0.6, s=50, edgecolors='black')
    
    # Highlight support vectors with different sizes based on alpha values
    support_vectors = model.support_vectors_
    ax.scatter(support_vectors[:, 0], support_vectors[:, 1], 
              s=200, facecolors='none', edgecolors='green', linewidth=3,
              label=f'Support Vectors ({len(support_vectors)})')
    
    # Plot decision boundary
    h = 0.02
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                        np.arange(y_min, y_max, h))
    
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], alpha=0.5,
              linestyles=['--', '-', '--'], colors=['red', 'black', 'red'])
    
    ax.set_title(f'{name}\\nSV Ratio: {len(support_vectors)/len(X_train):.3f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance summary
print("\\n🏆 FINAL PERFORMANCE RANKING:")
print("=" * 35)
sorted_results = sorted(results_comparison.items(), key=lambda x: x[1]['accuracy'], reverse=True)
for i, (name, results) in enumerate(sorted_results, 1):
    print(f"{i}. {name:20s}: {results['accuracy']:.3f}")

print("\\n💡 KEY INSIGHTS:")
print("=" * 20)
print("• SVM with RBF kernel excels at complex, non-linear boundaries")
print("• Linear SVM is efficient for linearly separable data")
print("• Support vectors represent the most informative training samples")
print("• Lower support vector ratio indicates better generalization")
print("• Kernel choice significantly impacts decision boundary shape")

In [ ]:
# 📈 2. HYPERPARAMETER OPTIMIZATION FOR RBF KERNEL
print("\n📈 2. RBF Kernel Hyperparameter Analysis")
print("-" * 50)

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Use the circular dataset for RBF kernel optimization
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)
scaler = StandardScaler()
X_circles_scaled = scaler.fit_transform(X_circles)
X_train, X_test, y_train, y_test = train_test_split(
    X_circles_scaled, y_circles, test_size=0.3, random_state=42)

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale', 'auto']
}

# Perform grid search
print("🔍 Performing Grid Search for optimal C and gamma...")
grid_search = GridSearchCV(SVC(kernel='rbf'), param_grid, 
                          cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"✅ Best parameters: {grid_search.best_params_}")
print(f"✅ Best cross-validation score: {grid_search.best_score_:.3f}")

# Visualize parameter effects
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Heatmap of parameter combinations
C_values = param_grid['C']
gamma_values = [g for g in param_grid['gamma'] if isinstance(g, float)]
scores_matrix = np.zeros((len(C_values), len(gamma_values)))

for i, C in enumerate(C_values):
    for j, gamma in enumerate(gamma_values):
        svm_temp = SVC(kernel='rbf', C=C, gamma=gamma)
        svm_temp.fit(X_train, y_train)
        scores_matrix[i, j] = svm_temp.score(X_test, y_test)

im = axes[0].imshow(scores_matrix, cmap='viridis', aspect='auto')
axes[0].set_xticks(range(len(gamma_values)))
axes[0].set_xticklabels([f'{g:.3f}' for g in gamma_values])
axes[0].set_yticks(range(len(C_values)))
axes[0].set_yticklabels(C_values)
axes[0].set_xlabel('Gamma')
axes[0].set_ylabel('C')
axes[0].set_title('Parameter Grid Accuracy Heatmap')
plt.colorbar(im, ax=axes[0], label='Accuracy')

# Plot 2: Effect of C parameter
C_range = np.logspace(-2, 2, 20)
train_scores = []
test_scores = []

for C in C_range:
    svm = SVC(kernel='rbf', C=C, gamma='scale')
    svm.fit(X_train, y_train)
    train_scores.append(svm.score(X_train, y_train))
    test_scores.append(svm.score(X_test, y_test))

axes[1].semilogx(C_range, train_scores, 'o-', label='Training accuracy', alpha=0.8)
axes[1].semilogx(C_range, test_scores, 's-', label='Test accuracy', alpha=0.8)
axes[1].set_xlabel('C (Regularization parameter)')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Effect of C Parameter')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Effect of gamma parameter
gamma_range = np.logspace(-4, 1, 20)
train_scores = []
test_scores = []

for gamma in gamma_range:
    svm = SVC(kernel='rbf', C=1.0, gamma=gamma)
    svm.fit(X_train, y_train)
    train_scores.append(svm.score(X_train, y_train))
    test_scores.append(svm.score(X_test, y_test))

axes[2].semilogx(gamma_range, train_scores, 'o-', label='Training accuracy', alpha=0.8)
axes[2].semilogx(gamma_range, test_scores, 's-', label='Test accuracy', alpha=0.8)
axes[2].set_xlabel('Gamma')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Effect of Gamma Parameter')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Train final model with best parameters
best_svm = grid_search.best_estimator_
y_pred = best_svm.predict(X_test)

print("\n📊 DETAILED PERFORMANCE METRICS:")
print("=" * 40)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Class 0', 'Class 1'],
            yticklabels=['Class 0', 'Class 1'])
plt.title('Confusion Matrix - Optimized RBF SVM')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print(f"\n🎯 Final Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"🔧 Number of Support Vectors: {len(best_svm.support_vectors_)}")
print(f"📏 Support Vector Ratio: {len(best_svm.support_vectors_)/len(X_train):.3f}")